# Phase 4 — Pooling for query types 3 and 4

Run **BM25** and **TF-IDF** on semantic (type 3) and relational (type 4) queries, merge the top-k results into a **judging pool**, enrich rows with **caption** and **schema** from `documents.jsonl`, and export **Excel** for manual relevance labels.

**After labeling:** fill `relevant` (e.g. `1` / `0` or `yes` / `no`), then convert to `data/qrels/qrels_type_3.json` and `qrels_type_4.json` (separate step / script).

**Requirements:** built indices under `models/bm25/` and `models/tfidf/` (run `scripts/build_index.py` first).

In [ ]:
# Setup — paths and imports
import json
import os
import re
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 30)


def find_root() -> Path:
    marker = Path("data") / "raw_data"
    for key in ("IR_PROJECT_ROOT", "VSCODE_WORKSPACE_FOLDER", "CURSOR_WORKSPACE_FOLDER"):
        v = os.environ.get(key)
        if v and (Path(v) / marker).exists():
            return Path(v)
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    nb = globals().get("__vsc_ipynb_file__") or os.environ.get("JPY_SESSION_NAME", "")
    if nb:
        for p in Path(nb).parents:
            if (p / marker).exists():
                return p
    raise FileNotFoundError("Cannot find project root. Set IR_PROJECT_ROOT.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

QUERIES_DIR = ROOT / "data" / "queries"
MODELS_DIR = ROOT / "models"
DOCS_FULL = ROOT / "data" / "json_format_data" / "full" / "documents.jsonl"
DOCS_SUB = ROOT / "data" / "json_format_data" / "subset_100k" / "documents.jsonl"
DOCS_PATH = DOCS_FULL if DOCS_FULL.exists() else DOCS_SUB
OUT_DIR = ROOT / "data" / "pooling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT      : {ROOT}")
print(f"Queries   : {QUERIES_DIR.exists()}")
print(f"BM25      : {(MODELS_DIR / 'bm25' / 'index.pkl').exists()}")
print(f"TF-IDF    : {(MODELS_DIR / 'tfidf' / 'matrix.npz').exists()}")
print(f"Documents : {DOCS_PATH}  exists={DOCS_PATH.exists()}")
print(f"Output    : {OUT_DIR}")


## Load retrievers

Loads pickled indices from `models/` (RAM-heavy: ~3–5 GB for BM25 corpus stats + sparse matrix for TF-IDF).

In [ ]:
from src.retrieval.classical_ir import BM25Retriever, TFIDFRetriever

bm25 = BM25Retriever()
bm25.load(MODELS_DIR / "bm25")

tfidf = TFIDFRetriever()
tfidf.load(MODELS_DIR / "tfidf")


## Pooling helpers

- **BM25 query tokens:** word tokens with the same boundary rule as the TF-IDF vectoriser (`\b\w+\b`), lowercased.
- **Pool:** union of top-`k` from each system per query; wide columns for rank/score and empty `relevant` for judging.

In [ ]:
TOKEN_PATTERN = re.compile(r"(?u)\b\w+\b")


def query_tokens_bm25(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


def load_queries_json(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)
    return payload["queries"]


def pool_one_query(
    bm25: BM25Retriever,
    tfidf: TFIDFRetriever,
    query_id: str,
    query_type: int,
    primary_text: str,
    notes: str,
    k: int,
) -> list[dict]:
    b_hits = bm25.search(query_tokens_bm25(primary_text), k=k)
    t_hits = tfidf.search(primary_text, k=k)

    b_rank = {doc_id: r + 1 for r, (doc_id, _) in enumerate(b_hits)}
    b_score = dict(b_hits)
    t_rank = {doc_id: r + 1 for r, (doc_id, _) in enumerate(t_hits)}
    t_score = dict(t_hits)

    all_ids = list(dict.fromkeys(list(b_rank.keys()) + list(t_rank.keys())))

    rows = []
    for did in all_ids:
        rows.append(
            {
                "query_id": query_id,
                "query_type": query_type,
                "query_text": primary_text,
                "query_notes": notes,
                "doc_id": did,
                "bm25_rank": b_rank.get(did, ""),
                "bm25_score": b_score.get(did, ""),
                "tfidf_rank": t_rank.get(did, ""),
                "tfidf_score": t_score.get(did, ""),
                "in_bm25": int(did in b_rank),
                "in_tfidf": int(did in t_rank),
                "relevant": "",
                "judge_notes": "",
            }
        )
    return rows


def build_pool_dataframe(
    bm25: BM25Retriever,
    tfidf: TFIDFRetriever,
    queries: list[dict],
    k: int = 100,
) -> pd.DataFrame:
    all_rows: list[dict] = []
    for q in queries:
        qid = q["query_id"]
        qtype = int(q["query_type"])
        texts = q.get("query_texts") or []
        primary = texts[0] if texts else ""
        notes = q.get("notes", "")
        if not primary.strip():
            continue
        all_rows.extend(
            pool_one_query(bm25, tfidf, qid, qtype, primary, notes, k)
        )
    return pd.DataFrame(all_rows)


def enrich_captions(df: pd.DataFrame, docs_path: Path) -> pd.DataFrame:
    """Single pass over documents.jsonl; fill caption + schema + text_blob preview."""
    need = set(df["doc_id"].astype(str))
    meta: dict[str, dict] = {}
    if not docs_path.exists():
        print(f"Warning: {docs_path} not found — skipping enrichment")
        df = df.copy()
        df["caption"] = ""
        df["schema"] = ""
        df["text_preview"] = ""
        return df

    print(f"Enriching {len(need):,} unique doc_ids from {docs_path.name} …")
    found = 0
    with open(docs_path, encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            did = doc["doc_id"]
            if did not in need:
                continue
            blob = doc.get("text_blob") or ""
            meta[did] = {
                "caption": doc.get("caption", ""),
                "schema": doc.get("schema", ""),
                "text_preview": blob[:500] + ("…" if len(blob) > 500 else ""),
            }
            found += 1
            if found >= len(need):
                break

    out = df.copy()
    out["caption"] = out["doc_id"].map(lambda x: meta.get(x, {}).get("caption", ""))
    out["schema"] = out["doc_id"].map(lambda x: meta.get(x, {}).get("schema", ""))
    out["text_preview"] = out["doc_id"].map(
        lambda x: meta.get(x, {}).get("text_preview", "")
    )
    missing = need - set(meta.keys())
    if missing:
        print(f"Warning: {len(missing)} doc_ids not found in corpus")
    return out


## Run pooling and export Excel

Adjust **`K_PER_SYSTEM`** (depth per retriever) and re-run. Output files:

- `data/pooling/pool_type_3.xlsx`
- `data/pooling/pool_type_4.xlsx`

In [ ]:
K_PER_SYSTEM = 100

for qtype, fname in ((3, "queries_type_3.json"), (4, "queries_type_4.json")):
    qpath = QUERIES_DIR / fname
    if not qpath.exists():
        print(f"Skip (missing): {qpath}")
        continue
    queries = load_queries_json(qpath)
    df = build_pool_dataframe(bm25, tfidf, queries, k=K_PER_SYSTEM)
    df = enrich_captions(df, DOCS_PATH)
    out_xlsx = OUT_DIR / f"pool_type_{qtype}.xlsx"
    df.to_excel(out_xlsx, index=False, sheet_name="Pool")
    print(f"Wrote {out_xlsx.name}: {len(df):,} rows, {df['query_id'].nunique()} queries")

df.head(8)
